In [1]:
import numpy as np
import cv2

In [2]:
corner_track_params = dict(maxCorners=10, qualityLevel=0.3, minDistance=7, blockSize=7)
lk_params = dict(winSize=(200, 200), maxLevel=2, criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))

In [3]:
cap = cv2.VideoCapture(0)
ret, old_frame = cap.read()
prev_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)

# pts to track
prev_pts = cv2.goodFeaturesToTrack(prev_gray, mask=None, **corner_track_params)

mask = np.zeros_like(old_frame)

while True:
    ret, frame = cap.read()

    frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    nextPts, status, err = cv2.calcOpticalFlowPyrLK(prev_gray, frame_gray, prev_pts, None, **lk_params)

    # Select good points
    good_new = nextPts[status == 1]
    good_prev = prev_pts[status == 1]

    for i, (new, prev) in enumerate(zip(good_new, good_prev)):

        # Draw the tracks
        x_new, y_new = new.ravel()
        x_prev, y_prev = prev.ravel()

        # Draw line between previous and new points
        mask = cv2.line(mask, (int(x_new), int(y_new)), (int(x_prev), int(y_prev)), (0, 255, 0), 3)

        # Draw current point
        frame = cv2.circle(frame, (int(x_new), int(y_new)), 5, (0, 0, 255), -1)
 
    # Add the mask to the frame
    img = cv2.add(frame, mask)
    cv2.imshow("tracking", img)

    # Wait for the ESC key to exit
    k= cv2.waitKey(30) & 0xff
    if k == 27:
        break

# Update previous frame and previous points
    prev_gray = frame_gray.copy()
    old_pts = good_new.reshape(-1, 1, 2)

cv2.destroyAllWindows()
cap.release()